# Download datasets

### Note the datasets could be very large! Feel free to check the [dataset website](https://mengtingwan.github.io/data/goodreads) and download the files you really need.

## Download in python

**We can also install the gdown package here and download files in python**

In [1]:
import requests
import gzip
import shutil
import time
import pandas as pd
from pathlib import Path
from requests.exceptions import ChunkedEncodingError, ConnectionError, ReadTimeout

**Specify your directory here:**

In [2]:
DIR = Path.cwd()

**Load data types and names**

In [3]:
file_names = pd.read_csv(DIR / 'dataset_names.csv')
display(file_names)

,type,name
0,complete,goodreads_book_works.json.gz
1,complete,goodreads_book_authors.json.gz
2,complete,goodreads_book_series.json.gz
3,complete,goodreads_books.json.gz
4,complete,goodreads_book_genres_initial.json.gz
5,complete,book_id_map.csv
6,complete,user_id_map.csv
7,complete,goodreads_interactions.csv
8,complete,goodreads_reviews_spoiler.json.gz
9,complete,goodreads_reviews_spoiler_raw.json.gz


**Now we can construct the urls to download files by name**

In [4]:
file_name_type_mapping = dict(zip(file_names['name'].values, file_names['type'].values))
file_name_url_mapping = {}

for fname in file_name_type_mapping:
    ftype = file_name_type_mapping[fname]
    if ftype == "complete":
        url = 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/'+fname
        file_name_url_mapping[fname] = url
    elif ftype == "byGenre":
        url = 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/'+fname
        file_name_url_mapping[fname] = url

In [5]:
def download_by_name(fname, local_filename, max_retries=3):
    is_zipped = local_filename.suffix == '.gz'
    unzipped_local_filename = local_filename.with_suffix('') if is_zipped else local_filename
    
    if unzipped_local_filename.exists():
        print('Dataset', unzipped_local_filename, 'has been downloaded!')
        return

    if fname in file_name_url_mapping:
        url = file_name_url_mapping[fname]

        download_success = False
        for attempt in range(max_retries):
            try:
                with requests.get(url, stream=True, timeout=30) as r:
                    r.raise_for_status()
                    with open(local_filename, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            f.write(chunk)
                download_success = True
                break

            except (ChunkedEncodingError, ConnectionError, ReadTimeout) as e:
                if attempt < max_retries - 1:
                    time.sleep(5)
                else:
                    print('Dataset', local_filename, 'has failed to download')

        if download_success:
            if is_zipped:
                with gzip.open(local_filename, 'rb') as f_in:
                    with open(unzipped_local_filename, 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)

                local_filename.unlink()

            print('Dataset', unzipped_local_filename, 'has been downloaded')
    else:
        print('Dataset', fname, 'can not be found!')

In [6]:
OUT_DIR = DIR.parent / 'data'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for file_name in file_names.itertuples(index=False):
    outpath = OUT_DIR / file_name.name
    download_by_name(file_name.name, outpath)

Dataset /mnt/e/Code/Project Recommender System/data/goodreads_book_works.json has been downloaded
Dataset /mnt/e/Code/Project Recommender System/data/goodreads_book_authors.json has been downloaded
Dataset /mnt/e/Code/Project Recommender System/data/goodreads_book_series.json has been downloaded
Dataset /mnt/e/Code/Project Recommender System/data/goodreads_books.json has been downloaded
Dataset /mnt/e/Code/Project Recommender System/data/goodreads_book_genres_initial.json has been downloaded
Dataset /mnt/e/Code/Project Recommender System/data/book_id_map.csv has been downloaded
Dataset /mnt/e/Code/Project Recommender System/data/user_id_map.csv has been downloaded
Dataset /mnt/e/Code/Project Recommender System/data/goodreads_interactions.csv has been downloaded
Dataset /mnt/e/Code/Project Recommender System/data/goodreads_reviews_spoiler.json has been downloaded
Dataset /mnt/e/Code/Project Recommender System/data/goodreads_reviews_spoiler_raw.json has been downloaded
Dataset /mnt/e/Cod